# LOB RL Training on Colab GPU

Train PPO agent for optimal execution in limit order book environment.

**Your data:** 28 CSV files in Google Drive `/csv` folder (512 MB)

**This notebook:**
- ✅ Automatically mounts your Google Drive
- ✅ Auto-splits data 80/20 (train/test)
- ✅ Trains on GPU (T4)
- ✅ Saves models to Drive (persistent)

In [ ]:
# Cell 1: Check GPU availability
# Should show: Tesla T4 or similar

!nvidia-smi

In [ ]:
# Cell 2: Clone your repository
# Replace YOUR_USERNAME with your GitHub username

!git clone https://github.com/YOUR_USERNAME/lob-sim-orderbook.git
%cd lob-sim-orderbook

In [ ]:
# Cell 3: Install dependencies (~2 minutes)

!pip install -e .
!pip install stable-baselines3[extra] wandb tensorboard

In [ ]:
# Cell 4: Mount Google Drive and access your CSV data
# Click the authorization link and allow access

from google.colab import drive
drive.mount('/content/drive')

# Create symlink to your data folder
!mkdir -p data
!ln -sf /content/drive/MyDrive/csv data/csv

# Verify data is accessible
print("First 10 files in your Drive /csv folder:")
!ls -lh data/csv/ | head -10

from pathlib import Path
n_csv = len(list(Path('data/csv').glob('*.csv')))
print(f"\n✅ Found {n_csv} CSV files")

In [ ]:
# Cell 5: Create 80/20 train/test split AUTOMATICALLY
# Works with ANY number of CSV files!

import os
from pathlib import Path

# Get all CSV files (sorted by name)
csv_files = sorted(list(Path('data/csv').glob('*.csv')))
n_files = len(csv_files)

# Calculate 80/20 split (rounded down)
n_train = int(n_files * 0.8)
n_test = n_files - n_train

print("=" * 60)
print("AUTOMATIC TRAIN/TEST SPLIT")
print("=" * 60)
print(f"Total CSV files: {n_files}")
print(f"Train files (80%): {n_train}")
print(f"Test files (20%): {n_test}")
print()

# Create directories
!mkdir -p data/train data/test

# Create symlinks for train set (first 80%)
print("Creating train set...")
for f in csv_files[:n_train]:
    target = Path(f'data/train/{f.name}')
    if not target.exists():
        os.symlink(f, target)

# Create symlinks for test set (remaining 20%)
print("Creating test set...")
for f in csv_files[n_train:]:
    target = Path(f'data/test/{f.name}')
    if not target.exists():
        os.symlink(f, target)

# Verify
train_count = len(list(Path('data/train').glob('*.csv')))
test_count = len(list(Path('data/test').glob('*.csv')))

print(f"\n✅ Train set: {train_count} files")
print(f"✅ Test set: {test_count} files")
print("\nTrain files (older data):")
!ls data/train/ | head -5
print("\nTest files (newer data):")
!ls data/test/

# Set variables for training cell
TRAIN_DATA = "data/train"
TEST_DATA = "data/test"

print(f"\n✅ Ready to train on {n_train} files, test on {n_test} files")

In [ ]:
# Cell 6: Optional - Monitor training with TensorBoard
# Run this cell BEFORE starting training, keep it running

%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/lob_logs

In [ ]:
# Cell 7: Train the model!
# Configuration guide:
#   Quick test (10 mins):  --timesteps 50000  --net-arch 32 32
#   Small (20 mins):       --timesteps 100000 --net-arch 64 64
#   Medium (1 hour):       --timesteps 500000 --net-arch 128 64
#   Large (3 hours):       --timesteps 1000000 --net-arch 128 128 64

!python src/py/train_rl.py \
    --train-data data/train \
    --test-data data/test \
    --timesteps 100000 \
    --target-qty 100 \
    --net-arch 64 64 \
    --batch-size 64 \
    --n-steps 2048 \
    --eval-freq 10000 \
    --save-dir /content/drive/MyDrive/lob_models \
    --log-dir /content/drive/MyDrive/lob_logs

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)
print("Models saved to: /content/drive/MyDrive/lob_models/")
print("Logs saved to: /content/drive/MyDrive/lob_logs/")
print("\nYou can access these from your Google Drive anytime!")

In [ ]:
# Cell 8: View and download your trained models

print("Your saved models:")
!ls -lh /content/drive/MyDrive/lob_models/

print("\n" + "="*60)
print("OPTION 1: Access from Google Drive (recommended)")
print("="*60)
print("Go to drive.google.com")
print("Navigate to: lob_models/")
print("Download the folder to your computer")

print("\n" + "="*60)
print("OPTION 2: Download directly from Colab")
print("="*60)
print("Uncomment and run the code below:")
print()

# Uncomment to download:
# from google.colab import files
# !cd /content/drive/MyDrive && zip -r lob_models.zip lob_models/
# files.download('/content/drive/MyDrive/lob_models.zip')

In [ ]:
# Cell 9: Evaluate your trained model

!python src/py/train_rl.py \
    --eval-only \
    --model /content/drive/MyDrive/lob_models/best/best_model \
    --train-data data/test \
    --n-eval-episodes 20 \
    --target-qty 100

In [ ]:
# Cell 10: Compare with baselines (TWAP, VWAP, POV)

# Pick one of your test files
!python src/py/baselines.py \
    --data data/test/blockchain_l3_2025-01-01.csv \
    --strategy both \
    --qty 100